In [1]:
# 1.1

import pandas as pd # Importamos librería pandas para crear el DataFrame
import json # Importamos librería json para leer archivo de origen de datos

# Función
def creacion_fichero_genero(genre, año_inicio, año_fin):

    # Abrimos el JSON con los datos
    with open("definitivo.json", "r") as f:
        musica_json = json.load(f)

    # Creamos lista vacía para guardar los registros
    datos_genero = []

    # Recorremos cada diccionario "artista"
    for artista in musica_json:

        # Creamos las variables género y año
        genero = artista["genre"]
        año = artista["year"]
        
        # Filtramos los 5 géneros seleccionados y el rango de publicación (año)
        if genero == genre and año_inicio <= año <= año_fin:
            
            # Si se cumple el filtro, añadimos un nuevo diccionario
            datos_genero.append({
                "id": artista["id"],
                "artist_name": artista["artist_name"],
                "track_name": artista["track_name"],
                "genre": genero,
                "year": año
            })

    # Convertimos la lista de diccionarios en un DataFrame para facilitar la visualización de los datos y exportarlos
    data_music = pd.DataFrame(datos_genero)

    # Exportamos el DataFrame a CSV. orient="records" para estructurar el json y que sea fácil de leer, al igual que indent=4. force_ascii=False mantiene tíldes y caracteres especiales
    data_music.to_json(f"data_music_filtrado_{genre}.json", orient="records", indent=4, force_ascii=False)

In [2]:
creacion_fichero_genero("rock", 2020, 2025)
creacion_fichero_genero("latin", 2020, 2025)
creacion_fichero_genero("flamenco", 2020, 2025)
creacion_fichero_genero("rap", 2020, 2025)
creacion_fichero_genero("indie", 2020, 2025)

In [3]:
#1.2

import json # Para guardar el resultado final en JSON
import requests # Para llamar a la API Last.fm
import time # Para añadir pausas y no saturar la API

def info_artista_api (api_key, fichero_genero, genre):
    url = "https://ws.audioscrobbler.com/2.0/" # URL Básica de la API

    # Abrimos el archivo JSON generado en 1.1
    with open(fichero_genero, "r", encoding="utf-8") as f:
        data_music_api = json.load(f)

    # Sacamos la lista de artistas únicos para no repetir llamadas innecesarias a la API
    artistas_unicos = list(set([item["artist_name"] for item in data_music_api if item["artist_name"]]))

    # Creamos un diccionario donde guardaremos la información obtenida de la API para cada artista
    info_artistas = {}

    # Recorremos cada artista único y hacemos una sola llamada por artista
    for artista in artistas_unicos:

        # Preparamos los parámetros como indica la documentación de la API
        parametros = {
            "method": "artist.getinfo",   # método que queremos usar
            "artist": artista,            # artista actual del bucle
            "api_key": api_key,           # mi clave
            "format": "json"              # respuesta en formato JSON
        }

        # Valores por defecto (por si no encuentra al artista o falla la petición)
        biografia = ""
        listeners = ""
        playcount = ""
        similares = ""

        # Hacemos la llamada a la API (con timeout para que no se quede colgado)
        try:
            response = requests.get(url, params=parametros, timeout=10)
            data_lastfm = response.json()
        except:
            # Si hay error (por ejemplo timeout), nos quedamos con valores por defecto y seguimos
            info_artistas[artista] = {
                "biografia": biografia,
                "listeners": listeners,
                "playcount": playcount,
                "similares": similares
            }
            time.sleep(0.05)
            continue

        # Comprobamos si la API encontró al artista. Si no existe la clave "artist", significa que no lo encontró
        if "artist" in data_lastfm:

            artista_data = data_lastfm["artist"]

            # Comprobamos que exista la biografía 
            if "bio" in artista_data and "summary" in artista_data["bio"]:
                biografia = artista_data["bio"]["summary"]

            # Comprobamos que exista la estadística de popularidad 
            if "stats" in artista_data:
                if "listeners" in artista_data["stats"]:
                    listeners = artista_data["stats"]["listeners"]
                if "playcount" in artista_data["stats"]:
                    playcount = artista_data["stats"]["playcount"]

            # Comprobamos que existan los artistas similares 
            if "similar" in artista_data and "artist" in artista_data["similar"]:
                
                nombres = [] # Creamos una lista vacía para almacenar los artistas similares
                
                # Recorremos la lista de artistas similares y los añadimos a la lista
                for a in artista_data["similar"]["artist"]:
                    nombres.append(a["name"])
                
                # Convertimos la lista en texto separado por comas
                similares = ", ".join(nombres)

        # Guardamos la información en el diccionario principal
        info_artistas[artista] = {
            "biografia": biografia,
            "listeners": listeners,
            "playcount": playcount,
            "similares": similares
        }

        # Para no saturar a la API, añadimos una pausa (más baja para que tarde menos)
        time.sleep(0.05)

    # Ahora recorremos todas las filas originales y añadimos la información correspondiente
    for item in data_music_api:

        # Sacamos el nombre del artista de esa fila
        artista = item["artist_name"]

        # Si el artista existe en el diccionario, añadimos su info; si no, dejamos valores por defecto
        if artista in info_artistas or artista != "":
            item.update(info_artistas[artista])
        else:
            item["biografia"] = ""
            item["listeners"] = ""
            item["playcount"] = ""
            item["similares"] = ""

    # Guardamos el nuevo archivo.
    # Con ensure_ascii = False nos aseguramos de que mantenga las tildes y ñ. Con encoding="utf-8" mantenemos tildes.
    with open(f"data_music_matcheado_{genre}.json", "w", encoding="utf-8") as f:
        json.dump(data_music_api, f, indent=4, ensure_ascii=False)

In [ ]:
api_key_flamenco = "d79b38ce7ccd273a5e94f02f8dd3a299"
info_artista_api(api_key_flamenco, "data_music_filtrado_flamenco.json", "flamenco")

In [ ]:
api_key_indie = "6786e294b805290158a20e4430160911" # Clave de Last.fm
info_artista_api(api_key_indie, "data_music_filtrado_indie.json", "indie")

In [ ]:
api_key_rock = "484e0e00eb4ec9d3093d5fc446733ed0"
info_artista_api(api_key_rock, "data_music_filtrado_rock.json", "rock")

In [4]:
api_key_rap = "c46b71050be388c1720fb9983feb9494"
info_artista_api(api_key_rap, "data_music_filtrado_rap.json", "rap")

In [ ]:
api_key_latin = "9bb42ab8825458126cac1b7815e878e7"
info_artista_api(api_key_latin, "data_music_filtrado_latin.json", "latin")